# Stage 9: Pseudo-Cell Tokenization and GeneCompass Embedding

> **Outputs read: the active config** (printed by the first code cell). The default since 2026-11-15 is **refs5**, the
> registered rerun of Stages 8–14 after the 2026-11-11 audit (`docs/audit/remediation/STEP3_RESULTS.md`): the 15
> references rebuilt into `references_v5/` (HYPOTH re-annotated, the HEART merge applied), the Stage 8 exclusion list
> rebuilt in Ensembl 115, the target sum set by the registered formula, and the repaired gate, DE and network code.
> **Model: retrain v2** (phase 2 final). The default before refs5 (`_refs4all_rebuild` folders) is
> `config/profiles/refs4all_rebuild.yaml`; this notebook as executed on it is in git history at commit `12f94b9`
> (the 14-tissue refs4 version at `542b454`). Earlier comparisons: `deconvolution/ALL_TISSUE_RERUN_RESULTS.md`
> (refs4 against all tissues), `deconvolution/REFS4_RERUN_RESULTS.md` (July against refs4) and
> `docs/RETRAIN_V2_RESULTS.md` (April against v2).

## Stage Objective

Represent Stage 8 pseudo-cells in the fine-tuned GeneCompass latent space and verify that the resulting embeddings preserve analyzable biological variation.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Tokenization | Apply the Stage 5 rank-based procedure to pseudo-cell expression profiles. |
| 2 | Model inference | Generate CLS embeddings with the Stage 7 rat model. |
| 3 | Alignment validation | Verify row-level correspondence among expression, metadata, tokenized records, and embeddings. |
| 4 | Quality gate | Use Augur AUC to assess whether treatment-related structure is detectable by tissue and cell type. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stage 8 pseudo-cells, Stage 3 tokens, Stage 4 medians, and the Stage 7 model. |
| Primary outputs | Tokenized pseudo-cell datasets, aligned metadata, CLS embeddings, and Augur quality metrics. |

## Result Overview

The stage creates a reproducible embedding representation for downstream differential and supervised subspace analyses.

The sections below present the computational evidence in execution order. Every figure is printed by a code cell from the recorded stage outputs or the config; the markdown describes methods and points to those cells rather than quoting numbers.

In [1]:
import os, json
from pathlib import Path

from nb_config import PROJECT
from nb_config import GC_DIR
from nb_config import tissues

print(f"{'Tissue':<10} {'dataset/':>12}  {'cell_embeddings.npy':>24}  {'tokenize_summary.json':>22}")
print("-" * 74)
total_emb = 0
for t in tissues:
    ds  = GC_DIR / t / "dataset"
    emb = GC_DIR / t / "embeddings" / "cell_embeddings.npy"
    tsj = GC_DIR / t / "tokenize_summary.json"
    n_shards = len(list(ds.glob("*.arrow"))) if ds.exists() else 0
    emb_mb   = emb.stat().st_size / 1e6 if emb.exists() else 0
    total_emb += emb_mb
    summary  = json.loads(tsj.read_text()) if tsj.exists() else {}
    n_cells  = summary.get("n_pseudocells", "?")
    print(f"{t:<10} {n_shards:>8} shards  {emb_mb:>16.1f} MB  ({n_cells} cells × 768 dims)")
print("-" * 74)
print(f"{'TOTAL':<10} {total_emb:>36.1f} MB  embeddings")


config: (default config)
Stage 8: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/results_refs5/motrpac
Stage 9: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/genecompass_input_v2_refs5
16 tissues: BAT, BLOOD, CORTEX, HEART, HIPPOC, HYPOTH, KIDNEY, LIVER, LUNG, OVARY, SKMGN, SKMVL, SMLINT, SPLEEN, TESTES, WATSC
Tissue         dataset/       cell_embeddings.npy   tokenize_summary.json
--------------------------------------------------------------------------
bat               1 shards               0.9 MB  (300 cells × 768 dims)
blood             1 shards               2.0 MB  (650 cells × 768 dims)
cortex            1 shards               1.8 MB  (600 cells × 768 dims)
heart             1 shards               2.3 MB  (750 cells × 768 dims)
hippoc            1 shards               1.7 MB  (550 cells × 768 dims)
hypoth            1 shards               1.7 MB  (550 cells × 768 dims)
kidney            1 shards               2.3 MB  (750 cells × 768 dims)
liver     

## Step 1 — Tokenize Pseudo-cells

`tokenize_pseudocells.py` reuses the same tokenizer as Stage 5 (`pipeline/05_tokenization/tokenize_corpus.py`) with one
key difference, a library-size normalization before the median divide:

| Parameter | Stage 5 (training corpus) | Stage 9 (pseudo-cells) |
|-----------|--------------------------|------------------------|
| Input | Raw scRNA-seq counts | BayesPrism Z (count-scale) |
| `normalize_total` target | none (raw counts / hybrid median) | `deconvolution.pseudocell_target_sum` (calibrated to Z values) |
| Then | ÷ hybrid median → log2(1+x) → rank → top-k | same |
| `species` | 2 (rat) | 2 (rat) |
| Gene space | GeneCompass rat vocabulary | Same |

**Why a target sum?** BayesPrism posterior Z matrices have a different absolute scale than raw scRNA-seq counts, so
they are first scaled to a fixed total. The rule is the registered formula (`docs/RETRAIN_V2_PREREGISTRATION.md`,
lines 155–164; `docs/audit/remediation/STEP3_PREREG.md` 3): one target sum for every tissue, at which LIVER's
pseudo-cells have the training corpus's median token value. For refs5 it was computed on the refs5 LIVER pseudo-cells
before any tissue was tokenized for analysis (`deconvolution/calibrate_target_sum.py`, job 12376341; record
`docs/audit/remediation/record/step3/stage8/target_sum.json`). The earlier default's value had been calibrated on the
July-reference LIVER pseudo-cells (job 12013898) and was not repeated when the references changed (audit finding
6.8). One target sum for 16 tissues is a compromise: the next cell prints the target sum in the config and each
tissue's median token value. Token selection does not depend on the target sum; only the token values do.

The `tokenize_summary.json` written per tissue records: number of cells, eligible genes, value distribution
statistics, and mean expressed sequence length.

In [2]:
import json, pandas as pd
from pathlib import Path

from nb_config import GC_DIR
from nb_config import tissues

# Shortest tokenized pseudo-cell per tissue, from its dataset (cells under 200 genes are dropped first)
from nb_config import min_token_length
min_len = {t: min_token_length(t) for t in tissues}
from nb_config import CFG
print(f"deconvolution.pseudocell_target_sum (config): {CFG['deconvolution'].get('pseudocell_target_sum')}")

rows = []
for t in tissues:
    f = GC_DIR / t / "tokenize_summary.json"
    s = json.loads(f.read_text()) if f.exists() else {}
    rows.append({
        "tissue":        t,
        "cells":         s.get("n_pseudocells",     "?"),
        "eligible genes":s.get("n_eligible_genes",  "?"),
        "median value":  round(s.get("value_stats",{}).get("median", float("nan")), 3),
        "p90 value":     round(s.get("value_stats",{}).get("p90",    float("nan")), 3),
        "mean tok len":  round(s.get("mean_expressed_length", float("nan")), 1),
        "min tok len (kept)": min_len.get(t, "?"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print("Corpus reference (v2 tokenized corpus, Stage 5): median token value 0.915")
n_drop = {t: json.loads((GC_DIR / t / "tokenize_summary.json").read_text()).get("n_dropped_lt_min_genes", 0) for t in tissues}
# top_n, target sum and species as each tissue's tokenize_summary.json recorded them (not typed in)
tok_set = {k: sorted({json.loads((GC_DIR / t / "tokenize_summary.json").read_text()).get(k) for t in tissues})
           for k in ("top_n", "target_sum", "species")}
fmt = lambda v: "/".join(f"{x:g}" for x in v)
print(f"All tissues: top_n={fmt(tok_set['top_n'])}, target_sum={fmt(tok_set['target_sum'])}, "
      f"species={fmt(tok_set['species'])}; cells dropped (< 200 genes): {sum(n_drop.values())} "
      f"({', '.join(f'{t} {k}' for t, k in n_drop.items() if k) or 'none'})")

/depot/reese18/apps/motrpac-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


deconvolution.pseudocell_target_sum (config): 6700
tissue  cells  eligible genes  median value  p90 value  mean tok len  min tok len (kept)
   bat    300           14102         0.868      1.861        2044.5                1008
 blood    650           14811         0.878      2.607        1655.0                 256
cortex    600           15306         1.027      1.961        2032.7                1380
 heart    750           15765         0.919      1.814        2048.0                2048
hippoc    550           15243         1.056      1.955        2030.9                1225
hypoth    550           13985         0.990      1.827        2045.0                1518
kidney    750           14833         1.023      1.923        2033.8                1142
 liver    400           12701         0.910      1.805        2048.0                2048
  lung   1028           15158         1.131      2.182        1735.2                 200
 ovary    432           14095         0.996      1.951     

In [3]:
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import GC_DIR
from nb_config import tissues

mean_len, median_val, p90_val = [], [], []
for t in tissues:
    s = json.loads((GC_DIR / t / "tokenize_summary.json").read_text())
    mean_len.append(s["mean_expressed_length"])
    median_val.append(s["value_stats"]["median"])
    p90_val.append(s["value_stats"]["p90"])

colors = plt.cm.tab10(np.linspace(0, 1, len(tissues)))
y = np.arange(len(tissues))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Stage 9 — Pseudo-cell Tokenization Statistics", fontsize=13, fontweight="bold")

# Left: mean expressed sequence length
bars = ax1.barh(y, mean_len, color=colors, edgecolor="white", height=0.7)
ax1.axvline(2048, color="red", lw=1.2, ls="--", label="top_n cap = 2048")
ax1.axvline(np.mean(mean_len), color="gray", lw=1, ls=":", alpha=0.8,
            label=f"mean {np.mean(mean_len):.0f}")
for bar, v in zip(bars, mean_len):
    ax1.text(v + 8, bar.get_y() + bar.get_height()/2,
             f"{v:.0f}", va="center", fontsize=8.5)
ax1.set_yticks(y)
ax1.set_yticklabels([t.upper() for t in tissues])
ax1.set_xlabel("Mean expressed token sequence length")
ax1.set_title("Mean Token Sequence Length per Tissue")
ax1.set_xlim(0, 2300)
ax1.legend(fontsize=8)

# Right: value stats per tissue (median + p90)
width = 0.35
bars2 = ax2.barh(y - width/2, median_val, width, label="median",
                 color=colors, edgecolor="white", alpha=0.85)
bars3 = ax2.barh(y + width/2, p90_val, width, label="p90",
                 color=colors, edgecolor="white", alpha=0.45)
ax2.axvline(0.915, color="navy", lw=1.2, ls="--", label="v2 corpus median 0.915")
ax2.set_yticks(y)
ax2.set_yticklabels([t.upper() for t in tissues])
ax2.set_xlabel(f"Normalized nonzero token value (after target_sum={CFG['deconvolution'].get('pseudocell_target_sum')})")
ax2.set_title("Token Value Distribution vs Corpus Reference")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Step 2 — GeneCompass CLS Embedding (GPU)

`embed_cells.py` loads the fine-tuned rat GeneCompass model (Stage 7 retrain v2, phase-2 final save; path in the
config's model section) and runs inference on the tokenized dataset to extract **CLS embeddings**:

| Property | Value |
|----------|-------|
| Model | Rat GeneCompass retrain v2 (phase 2 final) |
| Architecture | BERT, as in the model's `config.json` (notebook 7) |
| Embedding | Position-0 (CLS) hidden state, last layer |
| Species token | 2 (rat) |
| Device | CUDA (GPU, Gilbreth) |
| Output dtype | float32 numpy array |
| Output layout | Row = one pseudo-cell; row order = the tokenized dataset's order (`pseudocells.h5ad` minus cells dropped at tokenization) |

The CLS token aggregates information from all gene tokens in the sequence via the encoder's self-attention layers.
Its representation captures the cell's transcriptional state in the GeneCompass learned embedding space, on top of the
prior-knowledge embeddings the model was initialised with. In the v2 init, rat genes with a many-to-one human or mouse
ortholog (T3) start from that ortholog's rows, and the promoter and co-expression priors of rat-only (T4) genes are
zero.

**Row alignment note:** `embed_cells.py` preserves the row order of the tokenized dataset, and the `--n-cells` flag is
set to the full count (never subsampled), so `cell_embeddings.npy[i, :]` corresponds to row `i` of `dataset/`. The
shapes are printed below.

In [4]:
import numpy as np
from pathlib import Path

from nb_config import GC_DIR
from nb_config import tissues

print(f"{'Tissue':<10} {'Shape':>16}  {'File (MB)':>12}  {'L2 norm (mean)':>16}  {'Std (dim mean)':>16}")
print("-" * 76)
total_cells = 0
for t in tissues:
    f = GC_DIR / t / "embeddings" / "cell_embeddings.npy"
    if not f.exists():
        print(f"{t:<10}  MISSING")
        continue
    emb = np.load(f)
    l2  = np.linalg.norm(emb, axis=1).mean()
    std = emb.std(axis=0).mean()
    total_cells += emb.shape[0]
    print(f"{t:<10} {str(emb.shape):>16}  {f.stat().st_size/1e6:>10.1f} MB"
          f"  {l2:>14.3f}  {std:>14.4f}")
print("-" * 76)
print(f"{'TOTAL':<10} {total_cells:>10,} cells × 768 dims")


Tissue                Shape     File (MB)    L2 norm (mean)    Std (dim mean)
----------------------------------------------------------------------------
bat              (300, 768)         0.9 MB          20.064          0.0334
blood            (650, 768)         2.0 MB          20.015          0.0669
cortex           (600, 768)         1.8 MB          20.093          0.0744
heart            (750, 768)         2.3 MB          20.028          0.0450
hippoc           (550, 768)         1.7 MB          20.085          0.0693
hypoth           (550, 768)         1.7 MB          20.076          0.0599
kidney           (750, 768)         2.3 MB          20.066          0.0493
liver            (400, 768)         1.2 MB          20.065          0.0418
lung            (1028, 768)         3.2 MB          20.041          0.0743
ovary            (432, 768)         1.3 MB          20.052          0.0635
skmgn            (450, 768)         1.4 MB          20.037          0.0475
skmvl            (45

In [5]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import GC_DIR
from nb_config import tissues

l2_means, l2_stds, dim_stds = [], [], []
for t in tissues:
    emb = np.load(GC_DIR / t / "embeddings" / "cell_embeddings.npy")
    l2  = np.linalg.norm(emb, axis=1)
    l2_means.append(l2.mean())
    l2_stds.append(l2.std())
    dim_stds.append(emb.std(axis=0).mean())

colors = plt.cm.tab10(np.linspace(0, 1, len(tissues)))
y = np.arange(len(tissues))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Stage 9 — GeneCompass CLS Embedding Geometry (768-d)", fontsize=13, fontweight="bold")

# Left: L2 norm per tissue (mean ± std)
ax1.barh(y, l2_means, xerr=l2_stds, color=colors, edgecolor="white",
         height=0.7, error_kw=dict(ecolor="gray", capsize=3))
for i, (m, s) in enumerate(zip(l2_means, l2_stds)):
    ax1.text(m + s + 0.05, i, f"{m:.2f}±{s:.2f}", va="center", fontsize=8)
ax1.set_yticks(y)
ax1.set_yticklabels([t.upper() for t in tissues])
ax1.set_xlabel("L2 norm of 768-d embedding vector (mean ± std across cells)")
ax1.set_title("Embedding L2 Norm per Tissue")
ax1.set_xlim(0, max(l2_means) + max(l2_stds) + 1.5)

# Right: mean standard deviation across 768 dimensions
bars2 = ax2.barh(y, dim_stds, color=colors, edgecolor="white", height=0.7)
for bar, v in zip(bars2, dim_stds):
    ax2.text(v + 0.0003, bar.get_y() + bar.get_height()/2,
             f"{v:.4f}", va="center", fontsize=8.5)
ax2.set_yticks(y)
ax2.set_yticklabels([t.upper() for t in tissues])
ax2.set_xlabel("Mean std across 768 dimensions (intra-tissue variation)")
ax2.set_title("Embedding Dimension Variance\n(proxy for within-tissue diversity)")

plt.tight_layout()
plt.show()


## Embedding Quality Gate — Augur AUC

After embedding, the Aim 2 gate checks whether the GeneCompass embeddings carry biologically meaningful signal by running **Augur** (Skinnider et al. 2021) per tissue × cell type × condition. Augur trains a random forest to distinguish two groups within the embedding, returning an AUC where:

- **AUC = 0.5** → embeddings are uninformative for this contrast
- **AUC → 1.0** → embeddings cleanly separate the groups

Two conditions are tested:
| Condition | Groups |
|-----------|--------|
| `trained` | trained exercise vs. sedentary control |
| `sex` | male vs. female |

The `pca_control.tsv` compares three representations for the same Augur task:
| Representation | Description |
|----------------|-------------|
| `embed` | GeneCompass CLS embedding (Stage 9 output) |
| `pca` | Top PCs (PCA-50) of the same pseudo-cells' expression |
| `genes` | Direct gene expression of the same pseudo-cells |

An `auc_embed > auc_pca ≥ auc_genes` pattern would indicate the GeneCompass embedding adds discriminative power beyond linear dimensionality reduction of the same data.

Augur is run on the exported embeddings (`gate_rcv10/augur_results.tsv`). `pca_control.tsv` is read from `gate_rcv10/` and uses the gate's splits (10 stratified 5-fold splits, 10,000 permutations). The gate's own probe goes on in further batches for blocks whose class more permutations could still change, and a hotspot is the resulting class, not `q < 0.05` (`docs/audit/remediation/STEP3_PREREG.md` 5.2; notebook 11). Blocks that fail the §6 rule (median per-sample library below the threshold in `REFERENCE_RELABEL_PREREG.md` §6) are not analysed. TESTES and OVARY have no Augur score: with one sex, each block has too few control pseudo-cells for the number per arm that Augur subsamples.

In [6]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import GC_DIR
df = pd.read_csv(GC_DIR / "gate_rcv10" / "augur_results.tsv", sep="\t")

# Focus on embed representation, trained condition
embed_trained = (
    df[(df.representation == "embed") & (df.condition == "trained")]
    .groupby("tissue")["augur_auc"].mean()
    .reset_index()
)
embed_sex = (
    df[(df.representation == "embed") & (df.condition == "sex")]
    .groupby("tissue")["augur_auc"].mean()
    .reset_index()
)
pca_trained = (
    df[(df.representation == "pca") & (df.condition == "trained")]
    .groupby("tissue")["augur_auc"].mean()
    .reset_index()
)

from nb_config import tissues as tissues_ord
def get_auc(sub, t):
    row = sub[sub.tissue == t]
    return float(row.augur_auc.iloc[0]) if len(row) else float("nan")

embed_t = [get_auc(embed_trained, t) for t in tissues_ord]
embed_s = [get_auc(embed_sex,     t) for t in tissues_ord]
pca_t   = [get_auc(pca_trained,   t) for t in tissues_ord]

x = np.arange(len(tissues_ord))
w = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Stage 9 — Augur AUC: Exercise Responsiveness in Embeddings", fontsize=13, fontweight="bold")

# Left: embed vs pca for trained condition
ax1.bar(x - w/2, embed_t, w, label="embed (GeneCompass)", color="steelblue", edgecolor="white")
ax1.bar(x + w/2, pca_t,   w, label="pca (top-50 PCA)",    color="coral",     edgecolor="white")
ax1.axhline(0.5, color="gray", ls="--", lw=1, label="random (AUC=0.5)")
ax1.set_xticks(x)
ax1.set_xticklabels([t.upper() for t in tissues_ord], rotation=30, ha="right")
ax1.set_ylabel("Mean Augur AUC (across cell types)")
ax1.set_title("Trained vs. Sedentary\n(embed vs. PCA baseline)")
ax1.set_ylim(0.4, 1.0)
ax1.legend(fontsize=8)

# Right: embed trained vs sex (within embed representation)
ax2.bar(x - w/2, embed_t, w, label="trained vs sedentary", color="steelblue",  edgecolor="white")
ax2.bar(x + w/2, embed_s, w, label="male vs female",        color="mediumseagreen", edgecolor="white")
ax2.axhline(0.5, color="gray", ls="--", lw=1, label="random")
ax2.set_xticks(x)
ax2.set_xticklabels([t.upper() for t in tissues_ord], rotation=30, ha="right")
ax2.set_ylabel("Mean Augur AUC (GeneCompass embeddings)")
ax2.set_title("GeneCompass Embeddings:\nExercise vs Sex Signal per Tissue")
ax2.set_ylim(0.4, 1.0)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [7]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from nb_config import GC_DIR
# PCA control by the gate method (10 stratified 5-fold splits x 10,000 permutations)
pc = pd.read_csv(GC_DIR / "gate_rcv10" / "pca_control.tsv", sep="\t")
from nb_config import tissues as tissues_ord
representations = [
    ("auc_embed", "GeneCompass embedding", "#377eb8"),
    ("auc_pca", "PCA baseline", "#e76f51"),
    ("auc_genes", "Gene-expression baseline", "#2a9d8f"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True, constrained_layout=True)
fig.suptitle("Stage 9: Representation Comparison by Tissue", fontsize=13, fontweight="bold")

for ax, (column, title, color) in zip(axes, representations):
    data = [pc.loc[pc.tissue == tissue, column].dropna().values for tissue in tissues_ord]
    parts = ax.violinplot(data, positions=np.arange(len(tissues_ord)), showmedians=True, widths=0.8)
    for body in parts["bodies"]:
        body.set_facecolor(color); body.set_edgecolor("white"); body.set_alpha(0.75)
    parts["cmedians"].set_color("black")
    ax.axhline(0.5, color="gray", ls="--", lw=0.9)
    ax.set_xticks(np.arange(len(tissues_ord)))
    ax.set_xticklabels([t.upper() for t in tissues_ord], rotation=40, ha="right", fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Tissue")
    ax.set_ylim(0.3, 1.02)
axes[0].set_ylabel("Augur AUC across cell types")
plt.show()


## Stage Summary

Stage 9 tokenizes the pseudo-cells with the same ranking semantics used for model training, embeds them with the rat
GeneCompass checkpoint, and verifies row alignment. Augur diagnostics provide an independent measure of
treatment-signal recoverability. The next cell computes the headline numbers from this config's outputs: cell-type
separation (silhouette), the cells lost at tokenization, Augur's agreement with the PLS-1 probe, and whether the
embedding beats PCA-50. refs5 against the default before it is in `docs/audit/remediation/STEP3_RESULTS.md` (B3, B4); the earlier comparisons
are in `deconvolution/ALL_TISSUE_RERUN_RESULTS.md` and `deconvolution/REFS4_RERUN_RESULTS.md`.

In [8]:
import json, numpy as np, pandas as pd
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from datasets import load_from_disk
from IPython.display import Markdown, display
from nb_config import GC_DIR, GATE_DIR, tissues

# cell-type separation: silhouette (cosine) of each tissue's embeddings by cell type, as deconvolution/embed_qc.py
sil = {}
for t in tissues:
    e = np.load(GC_DIR / t / "embeddings" / "cell_embeddings.npy")
    ct = np.array(load_from_disk(str(GC_DIR / t / "dataset"))["cell_type"])
    sil[t] = silhouette_score(normalize(e), ct, metric="cosine")
sil = pd.Series(sil).sort_values()

# tokenization: pseudo-cells dropped under the minimum gene count
tok = {t: json.loads((GC_DIR / t / "tokenize_summary.json").read_text()) for t in tissues}
dropped = {t: s.get("n_dropped_lt_min_genes", 0) for t, s in tok.items() if s.get("n_dropped_lt_min_genes", 0)}
n_emb = sum(np.load(GC_DIR / t / "embeddings" / "cell_embeddings.npy", mmap_mode="r").shape[0] for t in tissues)

# gate statistics, as deconvolution/corroborate_summary.py computes them, over the analysable blocks
m = pd.read_csv(GATE_DIR / "corroboration_merged.tsv", sep="\t")
a = m[m["analysable"]]
s1 = a[["augur_embed_trained", "sup_trained_auc"]].dropna()
rho = spearmanr(s1.iloc[:, 0], s1.iloc[:, 1]).correlation
s2 = a[["auc_embed", "auc_pca"]].dropna(); d = s2["auc_embed"] - s2["auc_pca"]
p_w = wilcoxon(s2["auc_embed"], s2["auc_pca"]).pvalue
s3 = a[["augur_embed_trained", "augur_pca_trained"]].dropna()
# A hotspot is the gate's class (is_hotspot). Tables written before refs5 have no class column; there it is
# q_sup_trained < 0.05.
hot = m[m["is_hotspot"] == True] if "is_hotspot" in m.columns else m[m["q_sup_trained"] < 0.05]

display(Markdown(f"""
**Results for this config ({len(tissues)} tissues):**

- **Cell-type separation.** Silhouette by cell type (cosine, all pseudo-cells of each tissue): median {sil.median():.3f}
  over tissues; lowest {sil.index[0].upper()} {sil.iloc[0]:.3f}, {sil.index[1].upper()} {sil.iloc[1]:.3f}; highest
  {sil.index[-1].upper()} {sil.iloc[-1]:.3f}.
- **Tokenization.** {n_emb:,} pseudo-cells embedded; dropped under the minimum gene count:
  {", ".join(f"{t.upper()} {n}" for t, n in dropped.items()) or "none"}.
- **Augur.** In the {len(hot)} hotspots (the gate's class) Augur's AUC is at least {hot["augur_embed_trained"].min():.2f}.
  Its block AUCs follow the PLS-1 probe's at Spearman {rho:.3f} over {len(s1)} analysable blocks with both scores
  (no threshold is registered for this correlation); OVARY and TESTES have no Augur score.
- **Embedding vs PCA.** The embedding's PLS-1 AUC beats PCA-50 of the same pseudo-cells in {int((d > 0).sum())} of
  {len(d)} analysable blocks (median {d.median():+.3f}, Wilcoxon p {p_w:.2g}). Augur, a random forest, scores the
  embedding at a median {s3.iloc[:, 0].median():.3f} and PCA-50 at {s3.iloc[:, 1].median():.3f}.
"""))


**Results for this config (16 tissues):**

- **Cell-type separation.** Silhouette by cell type (cosine, all pseudo-cells of each tissue): median 0.851
  over tissues; lowest BAT 0.382, BLOOD 0.541; highest
  TESTES 0.988.
- **Tokenization.** 9,060 pseudo-cells embedded; dropped under the minimum gene count:
  LUNG 22.
- **Augur.** In the 11 hotspots (the gate's class) Augur's AUC is at least 0.75.
  Its block AUCs follow the PLS-1 probe's at Spearman 0.819 over 130 analysable blocks with both scores
  (no threshold is registered for this correlation); OVARY and TESTES have no Augur score.
- **Embedding vs PCA.** The embedding's PLS-1 AUC beats PCA-50 of the same pseudo-cells in 88 of
  150 analysable blocks (median +0.029, Wilcoxon p 0.013). Augur, a random forest, scores the
  embedding at a median 0.603 and PCA-50 at 0.621.


## Reproducibility Notes

### SLURM Job History

The default tree's Stage 9 ran with the repeat of Stage 8 in the per-tissue jobs 12377805–12377820 (commit `44332e6`,
no modified tracked file), and its gate (the probe, Augur and the PCA control) in the chain job 12377824 from the same
commit (`docs/audit/remediation/STEP3_RESULTS.md`, B3 and B4). The target sum was computed by job 12376341. Each
tissue's embedding provenance records how many checkpoint tensors loaded, and how many were missing or unexpected
(the guard for audit finding 6.7).

### Key Hyperparameters

The config's `deconvolution.pseudocell_target_sum` (printed above), the model's top-k and context from its
`config.json`, rat species identifier `2`, final-layer CLS embedding, and no pseudo-cell subsampling.

### Re-run Commands

The default config carries v2 with all 16 tissues, so:

```bash
python pipeline/run_stage9.py --label liver --device cuda
```

The default before refs5: `export PIPELINE_PROFILE=config/profiles/refs4all_rebuild.yaml` (its own exclusion list
and `pseudocell_target_sum`). The 14-tissue refs4 result: `config/profiles/refs4_14.yaml`. The assembled all-tissue tree:
`config/profiles/refs_v4all.yaml`. The July references: `config/profiles/july_refs.yaml`. The April model:
`config/profiles/april.yaml` (its own `pseudocell_target_sum`).

Tokenization and inference are deterministic for fixed inputs, model weights, software, and hardware (the 2026-11-02
clean rebuild of the earlier default reproduced every `cell_embeddings.npy` byte for byte; refs5's Stage 9 was run
once). Row `i` in `cell_embeddings.npy` corresponds to row `i`
of the tokenized dataset.

In [9]:
import numpy as np, json, hashlib
from pathlib import Path

from nb_config import GC_DIR
from nb_config import tissues

print("=== cell_embeddings.npy inventory ===")
all_ok = True
total_cells = 0
for t in tissues:
    emb_f = GC_DIR / t / "embeddings" / "cell_embeddings.npy"
    tok_f = GC_DIR / t / "tokenize_summary.json"
    if not emb_f.exists():
        print(f"  {t:<8}  MISSING embedding")
        all_ok = False
        continue
    emb = np.load(emb_f)
    expected_n = json.loads(tok_f.read_text())["n_pseudocells"] if tok_f.exists() else "?"
    shape_ok = "OK" if emb.shape[1] == 768 and emb.shape[0] == expected_n else "SHAPE MISMATCH"
    total_cells += emb.shape[0]
    md5 = hashlib.md5(emb_f.read_bytes()).hexdigest()
    print(f"  {t:<8}  {emb.shape[0]:>5,} x 768  [{shape_ok}]  md5={md5[:12]}...")

print(f"\nTotal embedded cells: {total_cells:,}")

print("\n=== tokenize_summary.json checksums ===")
for t in tissues:
    f = GC_DIR / t / "tokenize_summary.json"
    if f.exists():
        md5 = hashlib.md5(f.read_bytes()).hexdigest()
        print(f"  {t:<8}  {md5}")

print(f"\nAll outputs present: {all_ok}")


=== cell_embeddings.npy inventory ===
  bat         300 x 768  [OK]  md5=08c7dd27833c...
  blood       650 x 768  [OK]  md5=685307aa42a2...
  cortex      600 x 768  [OK]  md5=a009200015ad...
  heart       750 x 768  [OK]  md5=321fb4f2db73...
  hippoc      550 x 768  [OK]  md5=3fc441f488f0...
  hypoth      550 x 768  [OK]  md5=fc707e7322c0...
  kidney      750 x 768  [OK]  md5=c3e228e8225c...
  liver       400 x 768  [OK]  md5=1841be936b13...
  lung      1,028 x 768  [OK]  md5=9b2241ba43a1...
  ovary       432 x 768  [OK]  md5=2270a924ed4f...
  skmgn       450 x 768  [OK]  md5=1ff28d3e9d4a...
  skmvl       450 x 768  [OK]  md5=4b7eeb136edc...
  smlint      650 x 768  [OK]  md5=95d27a041341...
  spleen      700 x 768  [OK]  md5=dd8bfcd1cc91...
  testes      150 x 768  [OK]  md5=d158418c2210...
  watsc       650 x 768  [OK]  md5=b6d4313eb6b6...

Total embedded cells: 9,060

=== tokenize_summary.json checksums ===
  bat       666014faa8bd30a2da56bed6f49ab069
  blood     ea3cd4c3beb8feb8387